In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
from scipy.io import wavfile
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
# from huggingface_hub import snapshot_download

# snapshot_download(repo_id="its5Q/biggest-ru-book", 
#                   repo_type="dataset", local_dir="./biggest-ru-book")

In [3]:
# import tarfile

# def loop(files):
#     files, _ = files
#     for f in tqdm(files):
#         try:
#             with tarfile.open(f, "r") as tar:
#                 tar.extractall(path='biggest-ru-book')
#             os.remove(f)
#         except:
#             pass

In [4]:
# multiprocessing(glob('biggest-ru-book/*.tar'), loop, 20, returned = False)

In [5]:
files = glob('biggest-ru-book/*.json')
len(files)

548611

In [6]:
def loop(files):
    files, _ = files
    base = 'biggest-ru-book_audio'
    os.makedirs(base, exist_ok=True)
    data = []
    for f in tqdm(files):
        try:
            audio_f = f.replace('.json', '.mp3')
            if not os.path.exists(audio_f):
                continue
            with open(f) as fopen:
                d = json.load(fopen)

            speaker = d['speaker_name']
            t = d['text'].strip()
            if len(t) < 2:
                continue
            audio_filename = os.path.split(audio_f)[1]
            audio_filename = os.path.join(base, audio_filename)
            audio_np, sr = sf.read(audio_f)
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': f"{base}_{speaker}"
            })
        except:
            pass
    return data

In [7]:
data = loop((files[:1], 0))

100%|██████████| 1/1 [00:00<00:00, 23.59it/s]


In [8]:
data

[{'audio_filename': 'biggest-ru-book_audio/b0b5c4d8fcc2a34e.mp3',
  'text': 'Любой каприз за свой счет провернуть можешь.',
  'speaker': 'biggest-ru-book_audio_grigorii_metelitsa'}]

In [9]:
data = multiprocessing(files, loop, cores = 30)

100%|██████████| 18287/18287 [27:59<00:00, 10.89it/s]


In [10]:
len(data)

548392

In [11]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'audio_filename': 'biggest-ru-book_audio/b0b5c4d8fcc2a34e.mp3',
 'text': 'Любой каприз за свой счет провернуть можешь.',
 'speaker': 'biggest-ru-book_audio_grigorii_metelitsa'}

In [12]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'biggest-ru-book')

Creating parquet from Arrow format: 100%|██████████| 2/2 [00:00<00:00,  7.90ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):  91%|█████████ | 46.7MB / 51.3MB, 5.07MB/s  
Processing Files (0 / 1):  99%|█████████▉| 50.9MB / 51.3MB, 5.41MB/s  
Processing Files (1 / 1): 100%|██████████| 51.3MB / 51.3MB, 5.07MB/s  
Processing Files (1 / 1): 100%|██████████| 51.3MB / 51.3MB, 5.13MB/s  
New Data Upload: 100%|██████████| 51.3MB / 51.3MB, 5.13MB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:11<00:00, 11.19s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/1054d8faed4d7c97403b4256850b09e4213d4445', commit_message='Upload dataset', commit_description='', oid='1054d8faed4d7c97403b4256850b09e4213d4445', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [13]:
audio_files = [d['audio_filename'] for d in data]

with open('biggest-ru-book-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [18]:
# !zip -rq biggest-ru-book_audio.zip biggest-ru-book_audio

In [17]:
# !hf upload malaysia-ai/Multilingual-TTS biggest-ru-book_audio.zip --repo-type=dataset

In [21]:
# !zip -rq biggest-ru-book_audio_neucodec.zip biggest-ru-book_audio_neucodec

In [22]:
# !hf upload malaysia-ai/Multilingual-TTS biggest-ru-book_audio_neucodec.zip --repo-type=dataset